# Demo with the best model for email text autocomplete

This notebook provides an interactive demonstration of the fine-tuned `distilgpt2` for email text autocompletion.

The workflow of this notebook is:
1. Set up the Colab environment.
2. Load the fine-tuned Transformer model.
3. Define a Python function to handle generating autocomplete suggestions.
4. Create and launch an interactive web interface with Gradio.

Some of the references I've used:
- https://www.gradio.app/guides/quickstart
- https://colab.research.google.com/drive/1BAw8QGFNqeKf1V0E3TLCzWyV6NM_qv_i


## 1. Set-up

In [1]:
print("Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

import os
project_dir = '/content/drive/MyDrive/project'
print(f"\nChanging directory to: {project_dir}")
%cd {project_dir}

print("\nInstalling/Checking libraries...")
!pip install -U gradio transformers torch accelerate huggingface_hub -q
print("Libraries installed/checked.")

print("\nImporting modules...")
import torch
import gradio as gr # Import Gradio

try:
    from src.model import setup_transformer_model, transformer_autocomplete
    print("Successfully imported functions from src.model")
except ModuleNotFoundError:
    print("\nERROR: Could not find src/model.py.")
    print("Make sure you ran the '%cd' command correctly above to navigate to your 'project' directory.")
except Exception as e:
    print(f"\nAn error occurred importing from src.model: {e}")

print("\nSetup cell complete.")

Mounting Google Drive...
Mounted at /content/drive

Changing directory to: /content/drive/MyDrive/project
/content/drive/MyDrive/project

Installing/Checking libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. load fine-tuned model

In [2]:
model_path = "/content/drive/MyDrive/project/fine_tuned_distilgpt2_subset_test"

print(f"Attempting to load model from: {model_path}")

# load model and tokenizer
tokenizer, model, _ = setup_transformer_model(model_path)

if model and tokenizer:
    try:
        model.to(torch.device('cpu'))
        print("Model explicitly moved to CPU for demo.")
        model_loaded = True
        print("Fine-tuned model loaded successfully!")
    except Exception as e:
         print(f"Could not move model to CPU: {e}. Proceeding with default device.")
         model_loaded = True # Model loaded, just didn't move
else:
    print("!!! ERROR: Failed to load the fine-tuned model. Demo cannot run. !!!")
    model_loaded = False

Attempting to load model from: /content/drive/MyDrive/project/fine_tuned_distilgpt2_subset_test
Loading transformer model: /content/drive/MyDrive/project/fine_tuned_distilgpt2_subset_test
Using device: CPU
Tokenizer and model loaded successfully.
Model explicitly moved to CPU for demo.
Fine-tuned model loaded successfully!


## 3. Define Autocomplete Function for Interface

In [3]:
def get_autocomplete_suggestion(prompt_text):
    """
    Takes prompt text, runs the model, returns formatted HTML suggestion.
    """
    if not model_loaded:
        return "<p style='color:red;'>ERROR: Model not loaded.</p>"

    if not prompt_text.strip():
        return "<p style='color:orange;'>Please enter some text to get a suggestion.</p>"

    print(f"Gradio received prompt: '{prompt_text[:50]}...'")
    try:
        suggestion_text = transformer_autocomplete(
            tokenizer,
            model,
            prompt_text,
            max_new_tokens=40,
            device=model.device # CPU
        )

        print(f"Model generated suggestion: '{suggestion_text[:50]}...'")


        import html
        escaped_prompt = html.escape(prompt_text)
        return f"<b>{escaped_prompt}</b>{suggestion_text}" # Bold prompt + suggestion

    except Exception as e:
        print(f"Error during generation: {e}")
        return f"<p style='color:red;'>An error occurred during generation: {e}</p>"

print("Gradio helper function defined.")

Gradio helper function defined.


## 4. Create and Launch Gradio Demo Interface

In [4]:
if model_loaded:
    print("Launching Gradio interface...")

    # define interface components
    iface = gr.Interface(
        fn=get_autocomplete_suggestion,
        inputs=gr.Textbox(
            lines=7,
            label="Your Email Text:",  # as I did train for all text including title, address, and body. etc. Made it general
            placeholder="Enter the beginning of your email here...",
            value="Subject: Meeting Follow-up\n\nHi Team,\n\nPlease find attached" # Default sample to show
        ),
        outputs=gr.HTML( # Use HTML output to render bold text etc.
            label="Model Autocomplete Suggestion"
        ),
        title="Fine-Tuned Email Autocomplete Demo",
        description="Enter starting text for an email. The fine-tuned model (distilgpt2) will suggest a completion. (Based on Enron data subset fine-tuning)",
        allow_flagging='never'
    )

    iface.launch(share=True, debug=False)

else:
    print("Cannot launch demo - model not loaded.")

Launching Gradio interface...


/usr/local/lib/python3.11/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2de1aa8f3ad7c22f2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
